# Day 2 AM Goals:

1. Get your notebook finalised so that it outputs two clean csv files.
2. Make sure that your cleaning uses one or more functions. MUST HAVE: a function to enrich the data. A function that works out the difference in days between the date columns. 
3. Turn your notebook into an executable .py file (manually).

### 1. Get your notebook finalised so that it outputs two clean csv files.

In [10]:
# Import clean CSV files
import pandas as pd

BASE_DIRECTORY = Path.cwd()
CLEAN_DATA_DIRECTORY = BASE_DIRECTORY / "Clean_Data"
BOOKS_CLEAN_PATH = CLEAN_DATA_DIRECTORY / "library_books_clean.csv"
CUSTOMERS_CLEAN_PATH = CLEAN_DATA_DIRECTORY / "library_customers_clean.csv"

for clean_file in (BOOKS_CLEAN_PATH, CUSTOMERS_CLEAN_PATH):
    if not clean_file.exists():
        raise FileNotFoundError(
            f"Clean file not found: {clean_file}. "
            "Keep the notebook beside the Clean_Data folder."
        )

books_clean = pd.read_csv(BOOKS_CLEAN_PATH)
customers_clean = pd.read_csv(CUSTOMERS_CLEAN_PATH)

print(f"Books loaded: {books_clean.shape}")
print(f"Customers loaded: {customers_clean.shape}")
print("\nBooks preview:")
print(books_clean.head().to_string(index=False))
print("\nCustomers preview:")
print(customers_clean.head().to_string(index=False))



Books loaded: (13, 10)
Customers loaded: (8, 2)

Books preview:
  id               books book_checkout book_returned days_allowed_to_borrow  customer_id  loan_days  days_allowed  invalid_date_order  returned_late
 1.0 Catcher in the Rye     2023-02-20    2023-02-25                2 weeks          1.0        5.0            14               False          False
 6.0        Little Women    2023-04-02    2023-05-01                2 weeks          1.0       29.0            14               False           True
 9.0            Catch 22    2023-04-15    2023-04-16                2 weeks          7.0        1.0            14               False          False
10.0        Animal Farm     2023-04-20    2023-04-24                2 weeks          2.0        4.0            14               False          False
11.0                1984    2023-04-23    2023-04-27                2 weeks          8.0        4.0            14               False          False

Customers preview:
 customer_id  customer

In [3]:
# Load both clean files
import pandas as pd
books_path = next(
        file for file in csv_files
        if "books_clean" in file.name
)
customers_path = next(
        file for file in csv_files
        if "customers_clean" in file.name
)
books_df = pd.read_csv(books_path)
customers_df = pd.read_csv(customers_path)

### 2. Make sure that your cleaning uses one or more functions. MUST HAVE: a function to enrich the data. A function that works out the difference in days between the date columns. 

In [11]:
# Calculate the days    
def normalise_column_names(dataframe: pd.DataFrame) -> pd.DataFrame:
    # Return a copy with lowercase snake_case column names.
    cleaned = dataframe.copy()
    cleaned.columns = (
        cleaned.columns.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned

In [12]:
# Days between Dates
def add_days_between(
    dataframe: pd.DataFrame,
    start_date_column: str,
    end_date_column: str,
    output_column: str = "days_between",
) -> pd.DataFrame:
    # Enrich a DataFrame with the difference in days between two dates.
    enriched = dataframe.copy()
    start_dates = pd.to_datetime(enriched[start_date_column], errors="coerce")
    end_dates = pd.to_datetime(enriched[end_date_column], errors="coerce")
    enriched[start_date_column] = start_dates
    enriched[end_date_column] = end_dates
    enriched[output_column] = (end_dates - start_dates).dt.days.astype("Int64")
    return enriched

In [13]:
# Standardise the clean books file and add the loan duration.
def finalise_books(dataframe: pd.DataFrame) -> pd.DataFrame:
    
    books = normalise_column_names(dataframe)

    required_columns = {
        "id",
        "books",
        "book_checkout",
        "book_returned",
        "customer_id",
    }
    missing_columns = required_columns.difference(books.columns)
    if missing_columns:
        raise ValueError(f"Books file is missing columns: {sorted(missing_columns)}")

    books["books"] = books["books"].astype("string").str.strip()
    books["id"] = pd.to_numeric(books["id"], errors="coerce").astype("Int64")
    books["customer_id"] = pd.to_numeric(
        books["customer_id"], errors="coerce"
    ).astype("Int64")

    books = add_days_between(
        books,
        start_date_column="book_checkout",
        end_date_column="book_returned",
        output_column="loan_days",
    )

    if "days_allowed" in books.columns:
        books["days_allowed"] = pd.to_numeric(
            books["days_allowed"], errors="coerce"
        ).astype("Int64")
        books["returned_late"] = (
            books["loan_days"] > books["days_allowed"]
        ).astype("boolean")

    books["invalid_date_order"] = books["loan_days"].lt(0).astype("boolean")
    return books.drop_duplicates().reset_index(drop=True)

In [14]:
# Standardise the already-clean customers file.
def finalise_customers(dataframe: pd.DataFrame) -> pd.DataFrame:
    # Standardise the already-clean customers file.
    customers = normalise_column_names(dataframe)
    required_columns = {"customer_id", "customer_name"}
    missing_columns = required_columns.difference(customers.columns)
    if missing_columns:
        raise ValueError(
            f"Customers file is missing columns: {sorted(missing_columns)}"
        )

    customers["customer_id"] = pd.to_numeric(
        customers["customer_id"], errors="coerce"
    ).astype("Int64")
    customers["customer_name"] = (
        customers["customer_name"].astype("string").str.strip()
    )
    return customers.drop_duplicates().reset_index(drop=True)

In [16]:
# Run functions and output final clean CSV files
books_final = finalise_books(books_clean)
customers_final = finalise_customers(customers_clean)

books_final.to_csv(
    BOOKS_CLEAN_PATH,
    index=False,
    date_format="%Y-%m-%d",
)
customers_final.to_csv(
    CUSTOMERS_CLEAN_PATH,
    index=False,
)

print("Final clean files saved:")
print(f" - {BOOKS_CLEAN_PATH}")
print(f" - {CUSTOMERS_CLEAN_PATH}")
print(f"Final book rows: {len(books_final)}")
print(f"Final customer rows: {len(customers_final)}")

Final clean files saved:
 - c:\Users\Admin\Desktop\VT-DE5M5\Clean_Data\library_books_clean.csv
 - c:\Users\Admin\Desktop\VT-DE5M5\Clean_Data\library_customers_clean.csv
Final book rows: 13
Final customer rows: 8


In [17]:
# Validate the enrichment and saved clean files
books_check = pd.read_csv(
    BOOKS_CLEAN_PATH,
    parse_dates=["book_checkout", "book_returned"],
)
customers_check = pd.read_csv(CUSTOMERS_CLEAN_PATH)

calculated_days = (
    books_check["book_returned"] - books_check["book_checkout"]
).dt.days

assert BOOKS_CLEAN_PATH.exists()
assert CUSTOMERS_CLEAN_PATH.exists()
assert "loan_days" in books_check.columns
assert books_check["loan_days"].equals(calculated_days)
assert customers_check["customer_id"].is_unique

print("Validation passed.")
print(f"{BOOKS_CLEAN_PATH.name}: {len(books_check)} rows")
print(f"{CUSTOMERS_CLEAN_PATH.name}: {len(customers_check)} rows")

Validation passed.
library_books_clean.csv: 13 rows
library_customers_clean.csv: 8 rows


## 3. Turn your notebook into an executable .py file (manually).

In [18]:
# Save the notebook to open PowerShell
conversion_command = "jupyter nbconvert --to script eda.ipynb"
execution_command = "python eda.py"

print("Run these commands in PowerShell or Terminal:")
print(conversion_command)
print(execution_command)

Run these commands in PowerShell or Terminal:
jupyter nbconvert --to script eda.ipynb
python eda.py


## 4. Upload CSV clean files to a SQL

### Install Python packages in PowerShell
python -m pip install sqlalchemy pyodbc

In SMSS:
1. Select master
2. Run query:
    CREATE DATABASE [LibraryDB];
    GO

In [27]:
# Check ODBC driver

import pyodbc
available_drivers = pyodbc.drivers()
print(available_drivers)

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


In [32]:
# Local SQL

from pathlib import Path

import pandas as pd
from sqlalchemy import (
    Boolean,
    Date,
    Integer,
    String,
    create_engine,
    text,
)
from sqlalchemy.engine import URL

SERVER_NAME = r"localhost"
DATABASE_NAME = "LibraryDB"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string},
)

engine = create_engine(
    connection_url,
    fast_executemany=True,
)

print("SQLAlchemy engine created.")

SQLAlchemy engine created.


In [33]:
# Load clean files

clean_data_folder = Path.cwd() / "Clean_Data"

books_clean = pd.read_csv(
    clean_data_folder / "library_books_clean.csv",
    parse_dates=["book_checkout", "book_returned"],
)

customers_clean = pd.read_csv(
    clean_data_folder / "library_customers_clean.csv"
)

In [34]:
# Create the SQl tables

books_clean.to_sql(
    name="library_books_clean",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    dtype={
        "id": Integer(),
        "books": String(255),
        "book_checkout": Date(),
        "book_returned": Date(),
        "days_allowed_to_borrow": String(50),
        "customer_id": Integer(),
        "days_allowed": Integer(),
        "loan_days": Integer(),
        "invaled_date_order": Boolean(),
        "returned_late": Boolean(),
    },
)

customers_clean.to_sql(
    name="library_customers_clean",
    con=engine,
    schema="dbo",
    if_exists="replace",
    index=False,
    dtype={
        "customer_id": Integer(),
        "customer_name": String(255),
    },
)

print("Both SQL tables were created successfully.")

Both SQL tables were created successfully.


In [35]:
# Verify tables

with engine.connect() as connection:
    books_count = pd.read_sql(
        text("SELECT COUNT(*) AS row_count FROM dbo.library_books_clean"),
        connection,
    )

    customers_count = pd.read_sql(
        text("SELECT COUNT(*) AS row_count FROM dbo.library_customers_clean"),
        connection,
    )

print("Books table:")
print(books_count)

print("Customers table:")
print(customers_count)

Books table:
   row_count
0         13
Customers table:
   row_count
0          8


### Tables in SMSS
dbo.library_books_clean

dbo.library_customers_clean